In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold

class FacialKeypointsDataset(Dataset):
    def __init__(self, df, is_train=False):
        self.df = df.reset_index(drop=True)
        self.is_train = is_train
        # 左右对称关键点对调索引
        self.flip_pairs = [(0,2), (1,3), (4,6), (5,7), (8,10), (9,11),
                           (12,14), (13,15), (16,18), (17,19), (22,24), (23,25)]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = np.fromstring(row['Image'], sep=' ').reshape(96, 96).astype(np.float32)
        pts = row.drop('Image').values.astype(np.float32)

        # 50% 概率执行水平翻转增强 (仅限训练集)
        if self.is_train and np.random.rand() > 0.5:
            img = np.fliplr(img)
            pts[0::2] = 96.0 - pts[0::2] # 翻转 X 坐标
            for i, j in self.flip_pairs: # 交换左右特征点
                pts[i], pts[j] = pts[j], pts[i]

        # 归一化处理
        img = img.reshape(1, 96, 96) / 255.0

        # 构建 Mask：记录非缺失值位置，并将 NaN 填 0 防报错
        mask = ~np.isnan(pts)
        pts = np.nan_to_num(pts, nan=0.0)

        return torch.tensor(img.copy()), torch.tensor(pts), torch.tensor(mask.astype(np.float32))

def get_cv_dataloaders(csv_path='../data/raw/training.csv', n_splits=5, batch_size=32, seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)

    df = pd.read_csv(csv_path)
    loaders = []

    for train_idx, val_idx in KFold(n_splits=n_splits, shuffle=True, random_state=seed).split(df):
        train_loader = DataLoader(FacialKeypointsDataset(df.iloc[train_idx], is_train=True), batch_size, shuffle=True)
        val_loader = DataLoader(FacialKeypointsDataset(df.iloc[val_idx], is_train=False), batch_size, shuffle=False)
        loaders.append((train_loader, val_loader))

    print(f"成功生成 {n_splits} 折 DataLoader 列表！")
    return loaders

# 获取数据加载器
cv_loaders = get_cv_dataloaders()


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# -----------------------------------------
# 1. 定义 Baseline 模型: 简单 CNN
# -----------------------------------------
class SimpleCNNBaseline(nn.Module):
    def __init__(self):
        super(SimpleCNNBaseline, self).__init__()
        # 输入维度: (Batch_size, 1, 96, 96)
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # 输出: (32, 48, 48)

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # 输出: (64, 24, 24)

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # 输出: (128, 12, 12)
        )

        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 12 * 12, 512),
            nn.ReLU(),
            nn.Dropout(0.2), # 防止过拟合
            nn.Linear(512, 30) # 输出 30 个坐标值 (15个关键点 * 2)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

# -----------------------------------------
# 2. 定义带 Mask 的损失函数与 RMSE 计算
# -----------------------------------------
def masked_mse_loss(preds, targets, masks):
    """
    只计算有标签（mask==1）位置的均方误差
    """
    diff = preds - targets
    masked_diff = diff * masks # 缺失值对应的位置会被乘 0 忽略掉
    # 计算有效点的均方误差 (加上 1e-8 防止除以 0)
    loss = (masked_diff ** 2).sum() / (masks.sum() + 1e-8)
    return loss

def compute_rmse(preds, targets, masks):
    """用于评估的 RMSE (Kaggle 官方指标)"""
    mse = masked_mse_loss(preds, targets, masks)
    return torch.sqrt(mse)


In [ ]:
import torch
import torch.nn as nn
from torchvision import models

# -----------------------------------------
# 1. 定义改进模型: Pretrained MobileNetV2
# -----------------------------------------
class MobileNetFacialKeypoints(nn.Module):
    def __init__(self, pretrained=True):
        super(MobileNetFacialKeypoints, self).__init__()

        # 加载预训练的 MobileNetV2
        # 注意：权重选择当前推荐的 Weights 格式
        weights = models.MobileNet_V2_Weights.DEFAULT if pretrained else None
        self.model = models.mobilenet_v2(weights=weights)

        # 改进 1: 修改第一层卷积，使其接受 1 通道的灰度图 (Kaggle 数据是 96x96 灰度图)
        # 原始层是 nn.Conv2d(3, 32, ...)
        original_conv = self.model.features[0][0]
        self.model.features[0][0] = nn.Conv2d(
            in_channels=1,
            out_channels=original_conv.out_channels,
            kernel_size=original_conv.kernel_size,
            stride=original_conv.stride,
            padding=original_conv.padding,
            bias=False
        )

        # 改进 2: 修改最后的分类层 (Classifier) 变为回归层
        # MobileNetV2 的最后一部分是 Dropout + Linear(1280, 1000)
        num_ftrs = self.model.classifier[1].in_features
        self.model.classifier[1] = nn.Sequential(
            nn.Linear(num_ftrs, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 30) # 最终输出 30 个坐标点
        )

    def forward(self, x):
        return self.model(x)

# -----------------------------------------
# 2. 实例化模型并检查参数量 (满足项目 0.5B 限制要求)
# -----------------------------------------
improved_model = MobileNetFacialKeypoints(pretrained=True)
total_params = sum(p.numel() for p in improved_model.parameters())
print(f"改进模型总参数量: {total_params / 1e6:.2f} M (符合 < 500M 要求)")

In [ ]:
# -----------------------------------------
# 通用训练与评估引擎
# -----------------------------------------
def train_and_evaluate(loaders, model_class, model_name="Model", num_epochs=15, lr=1e-3):
    """
    参数:
    - model_class: 传入模型的类名 (不要加括号)
    - model_name: 打印日志用的名字
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"[{model_name}] 开始训练 | 当前计算设备: {device}")

    fold_rmses = []

    # 遍历 5 个折 (Fold)
    for fold_idx, (train_loader, val_loader) in enumerate(loaders):
        print(f"\n========== {model_name} | Fold {fold_idx + 1}/5 ==========")

        # 【核心修改】这里动态实例化模型，彻底解耦！
        model = model_class().to(device)
        optimizer = optim.Adam(model.parameters(), lr=lr)

        for epoch in range(num_epochs):
            # --- 训练阶段 ---
            model.train()
            train_loss_sum = 0.0

            for images, keypoints, masks in train_loader:
                images = images.to(device)
                keypoints = keypoints.to(device)
                masks = masks.to(device)

                optimizer.zero_grad()
                outputs = model(images)

                loss = masked_mse_loss(outputs, keypoints, masks)
                loss.backward()
                optimizer.step()

                train_loss_sum += loss.item()

            avg_train_loss = train_loss_sum / len(train_loader)

            # --- 验证阶段 ---
            model.eval()
            val_rmse_sum = 0.0

            with torch.no_grad():
                for images, keypoints, masks in val_loader:
                    images = images.to(device)
                    keypoints = keypoints.to(device)
                    masks = masks.to(device)

                    outputs = model(images)
                    batch_rmse = compute_rmse(outputs, keypoints, masks)
                    val_rmse_sum += batch_rmse.item()

            avg_val_rmse = val_rmse_sum / len(val_loader)

            # 打印进度
            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"Fold {fold_idx + 1} | Epoch [{epoch+1}/{num_epochs}] | Train MSE: {avg_train_loss:.4f} | Val RMSE: {avg_val_rmse:.4f}")

        fold_rmses.append(avg_val_rmse)
        print(f"-> {model_name} Fold {fold_idx + 1} 最终 Val RMSE: {avg_val_rmse:.4f}")

    # 计算 5 折的平均 RMSE
    final_cv_rmse = np.mean(fold_rmses)
    print(f"\n{'='*50}")
    print(f" {model_name} 5 折交叉验证平均 RMSE: {final_cv_rmse:.4f}")
    print(f"{'='*50}\n")

    return model, final_cv_rmse


In [ ]:
# -----------------------------------------
# 实验 A: 训练 Simple CNN Baseline
# -----------------------------------------
trained_base_model, baseline_score = train_and_evaluate(
    loaders=cv_loaders,
    model_class=SimpleCNNBaseline, # 传入基线模型类
    model_name="Baseline_CNN",
    num_epochs=15,
    lr=1e-3
)

In [ ]:
# -----------------------------------------
# 实验 B: 训练 Pretrained MobileNetV2
# -----------------------------------------
# 由于预训练模型有很好的底子，我们通常使用更小的学习率 (如 5e-4 或 1e-4) 进行微调
trained_improved_model, improved_score = train_and_evaluate(
    loaders=cv_loaders,
    model_class=MobileNetFacialKeypoints, # 传入改进模型类
    model_name="MobileNet_Pretrained",
    num_epochs=20,
    lr=5e-4
)